# Research data supporting "Practical and improved density functionals for computational catalysis on metal surfaces"

This notebook accompanies our paper: **Practical and improved density functionals for computational catalysis on metal surfaces**. It can be found on GitHub at https://github.com/benshi97/Data_NSCDFT_for_Metals and explored interactively on [Colab](https://colab.research.google.com/github/benshi97/Data_NSCDFT_for_Metals/blob/main/analyse.ipynb).

### Abstract

Density functional theory (DFT) has been critical towards our current atomistic understanding of catalysis on transition-metal surfaces.
It has opened new paradigms in rational catalyst design, predicting fundamental properties, like the adsorption energy and reaction barriers, linked to catalytic performance.
However, such applications depend sensitively on the predictive accuracy of DFT and achieving experimental-level reliability, so-called transition-metal chemical accuracy (13 kJ/mol), remains challenging for present density functional approximations (DFAs) or even methods beyond DFT.
We introduce a new framework for designing DFAs tailored towards studying molecules adsorbed on transition-metal surfaces, building upon recent developments in non-self-consistent DFAs.
We propose two functionals within this framework, demonstrating that transition-metal chemical accuracy can be achieved across a diverse set of 39 adsorption reactions while delivering consistent performance for 17 barrier heights.
Moreover, we show that these non-self-consistent DFAs address qualitative failures that challenge current self-consistent DFAs, such as CO adsorption on Pt(111) and graphene on Ni(111).
The resulting functionals are computationally practical and compatible with existing DFT codes, with scripts and workflows provided to use them.
Looking ahead, this framework establishes a path toward developing more accurate and sophisticated DFAs for computational heterogeneous catalysis and beyond.


## Table of Contents

1. [Lattice energies of X23 dataset](#lattice-energies-of-x23-dataset)
2. [Analysis DMC costs](#analysis-dmc-costs)
3. [Lattice and relative energies of ICE13](#lattice-and-relative-energies-of-ice13)
4. [Optimizing B86bPBE50-revXDM](#optimizing-b86bpbe50-revxdm)
5. [Relative energies of pharmaceutical polymorphs](#relative-energies-of-pharmaceutical-polymorphs)
6. [DFT Benchmarks](#dft-benchmarks)

In [1]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
    import os
    import subprocess
    import sys
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/Users/bshi':
        usetex = True
    else:
        usetex = False

if usetex:
    def textbf(text):
        return r'\textbf{' + text + '}'
else:
    def textbf(text):
        return text

# If in Google Colab, install the necessary data and set up the necessary environment
if IN_COLAB == True:
    repo_url = "https://github.com/benshi97/Data_NSCDFT_for_Metals"

    # Clone the repository
    %cd /content
    !rm -rf /content/Data_LNOMBECC
    !git clone {repo_url}
    %cd /content/Data_LNOMBECC

replot_analysis = False

# Import necessary functions
from Scripts.analysis_scripts import *
from Scripts.plot_scripts import *

if usetex == True:
    textrue_import()
else:
    texfalse_import()

dft_xc_list = [
    "01_PBE",
    "02_PBEsol",
    "03_RPBE",
    "04_revPBE",
    "05_BLYP",
    "06_r2SCAN",
    "07_MS2",
    "08_revTPSS",
    "09_PBED3",
    "10_PBEDDsC",
    "11_RPBED3",
    "12_optPBEvdW",
    "13_revvdWDF2",
    "14_r2SCANrVV10"
]

method_dict = {
    '01_PBE': {'label': 'PBE', 'color': 'orange'},
    '02_PBEsol': {'label': 'PBEsol', 'color': 'purple'},
    '03_RPBE': {'label': 'RPBE', 'color': 'cyan'},
    '04_revPBE': {'label': 'revPBE', 'color': 'magenta'},
    '05_BLYP': {'label': 'BLYP', 'color': 'lime'},
    '06_r2SCAN': {'label': r'r$^2$SCAN', 'color': 'pink'},
    '07_MS2': {'label': 'MS2', 'color': 'teal'},
    '08_revTPSS': {'label': 'revTPSS', 'color': 'lavendar'},
    '09_PBED3': {'label': 'PBE-D3', 'color': 'brown'},
    '10_PBEDDsC': {'label': 'PBE-DdsC', 'color': 'beige'},
    '11_RPBED3': {'label': 'RPBE-D3', 'color': 'maroon'},
    '12_optPBEvdW': {'label': 'optPBE-vdW', 'color': 'apricot'},
    '13_revvdWDF2': {'label': 'revvdW-DF2', 'color': 'navy'},
    '14_r2SCANrVV10': {'label': r'r$^2$SCAN+rVV10', 'color': 'olive'},
    'RPA': {'label': 'RPA@PBE', 'color': 'blue'},
    'BEEFvdW': {'label': 'BEEF-vdW', 'color': 'red'},
    'hBEEFvdW': {'label': 'hBEEF-vdW', 'color': 'green'},
    'dhBEEFvdW': {'label': 'dhBEEF-vdW', 'color': 'yellow'},
}


# CE39 dataset

## Experimental data

In [2]:
raw_data = { # Taken from Surf. Sci. 640, 36–44 (2015).
    "#": list(range(1, 40)),
    "Surface reaction": [
        "CO + Ni(111) → CO/Ni(111)",
        "CO + Pt(111) → CO/Pt(111)",
        "CO + Pd(111) → CO/Pd(111)",
        "CO + Pd(100) → CO/Pd(100)",
        "CO + Rh(111) → CO/Rh(111)",
        "CO + Ir(111) → CO/Ir(111)",
        "CO + Cu(111) → CO/Cu(111)",
        "CO + Ru(001) → CO/Ru(001)",
        "CO + Co(001) → CO/Co(001)",
        "NO + Ni(100) → N/Ni(100) + O/Ni(100)",
        "NO + Pt(111) → NO/Pt(111)",
        "NO + Pd(111) → NO/Pd(111)",
        "NO + Pd(100) → NO/Pd(100)",
        "O2 + Ni(111) → 2O/Ni(111)",
        "O2 + Ni(100) → 2O/Ni(100)",
        "O2 + Pt(111) → 2O/Pt(111)",
        "O2 + Rh(100) → 2O/Rh(100)",
        "H2 + Pt(111) → 2H/Pt(111)",
        "H2 + Ni(111) → 2H/Ni(111)",
        "H2 + Ni(100) → 2H/Ni(100)",
        "H2 + Rh(111) → 2H/Rh(111)",
        "H2 + Pd(111) → 2H/Pd(111)",
        "I2 + Pt(111) → 2I/Pt(111)",
        "CH2I2 + Pt(111) → CH/Pt(111) + H/Pt(111) + 2I/Pt(111)",
        "CH3I + Pt(111) → CH3/Pt(111) + I/Pt(111)",
        "NH3 + Cu(100) → NH3/Cu(100)",
        "CH3I + Pt(111) → CH3I/Pt(111)",
        "CH3OH + Pt(111) → CH3OH/Pt(111)",
        "CH4 + Pt(111) → CH4/Pt(111)",
        "C2H6 + Pt(111) → C2H6/Pt(111)",
        "C3H8 + Pt(111) → C3H8/Pt(111)",
        "C4H10 + Pt(111) → C4H10/Pt(111)",
        "C6H6 + Pt(111) → C6H6/Pt(111)",
        "C6H6 + Cu(111) → C6H6/Cu(111)",
        "C6H6 + Ag(111) → C6H6/Ag(111)",
        "C6H6 + Au(111) → C6H6/Au(111)",
        "C6H10 + Pt(111) → C6H10/Pt(111)",
        "H2O + Pt(111) → H2O/Pt(111)",
        "H2O + OPt(111) → H2OOH/Pt(111)"
    ],
    "Coverage (ML)": [
        "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/4", "1/8", "1/4", "1/4", "1/4", "1/8",
        "1/8", "1/18", "1/8", "1/8", "1/8", "1/8", "1/8", "1/8", "1/4", "1/4", "1/4", "1/4", "1/4",
        "1/4", "1/4", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/9", "1/4", "1/2"
    ],
    "Reaction enthalpy (kJ/mol)": [
        -122, -120, -143, -155, -139, -158, -53, -158, -115, -290, -114, -179, -161, -480,
        -530, -208, -358, -72, -94, -94, -70, -88, -312, -473, -212, -57, -84.5, -56, -15,
        -28.5, -41.3, -50.8, -164, -68, -63, -72, -122, -51.3, -57.4
    ],
    "Temp. (K)": [
        350, 340, 450, 430, 500, 420, 350, 475, 370, 300, 300, 520, 300, 300,
        300, 515, 300, 300, 370, 370, 325, 370, 0, 210, 320, 235, 100, 100,
        63, 106, 139, 171, 300, 225, 210, 230, 100, 120, 150
    ],
    "Reaction energy (kJ/mol)": [
        -119, -117, -139, -151, -135, -154, -52, -154, -112, -288, -112, -175, -159, -479,
        -528, -204, -356, -70, -91, -91, -67, -85, -230, -471, -209, -55, -83.7, -55,
        -14.5, -27.6, -40.1, -49.4, -300, -66, -61, -70, -121, -50.3, -56.2
    ],
    "Reaction energy without ZPE (kJ/mol)": [
        -124, -124, -144, -157, -142, -164, -57, -161, -119, -299,
        -119, -182, -163, -485, -530, -208, -355, -72, -100, -87,
        -72, -90, -313, -455, -209
    ] + [
        -60, -84, -55, -14, -27, -39, -48, -162, -66, -61,
        -70, -123, -55, -66
    ],
    "Weight": [ 1, 1, 1, 1, 1, 1, 1, 1, 1,0.5, 1, 1, 1, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5,0.5, 0.5, 0.5, 0.5, 0.25, 0.5] + [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,1]
}

ce39_dataset_dict = {}

calc_molecule_list = []
calc_molecule_surface_list = []

for ce39_idx in range(39):


    if 'Co' in raw_data['Surface reaction'][ce39_idx] or 'Ru' in raw_data['Surface reaction'][ce39_idx]:
        crystal_type = 'hcp'
    else:
        crystal_type = 'fcc'

    # Set the supercell_size based on coverage
    if raw_data['Coverage (ML)'][ce39_idx] in ['1/4','1/8']:
        surface_sc_size = '2x2'
    else:
        surface_sc_size = '3x3'

    surface_type = raw_data['Surface reaction'][ce39_idx].split()[2].replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')
    system_molecule = raw_data['Surface reaction'][ce39_idx].split()[0]
    molecule_surface_system_list = []
    for molecule_surface_system in raw_data['Surface reaction'][ce39_idx].split('→')[1].split('+'):
        if molecule_surface_system.strip()[0].isdigit():
            molecule_surface_system_list += [molecule_surface_system.strip()[1:].replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]*int(molecule_surface_system.strip()[0])
        else:
            molecule_surface_system_list += [molecule_surface_system.strip().replace('/','-').replace('(',f'_{crystal_type}_').replace(')',f'_{surface_sc_size}')]


    ce39_dataset_dict[f'{ce39_idx+1:02d}'] = {'Reaction': raw_data['Surface reaction'][ce39_idx], 'MAD Weight': raw_data['Weight'][ce39_idx], 'Reactant Molecule': [system_molecule], 'Reactant Surface': [surface_type], 'Products': molecule_surface_system_list, 'Reaction Enthalpy': int(raw_data["Reaction enthalpy (kJ/mol)"][ce39_idx]), 'Reaction Energy': int(raw_data["Reaction energy (kJ/mol)"][ce39_idx]),'Reaction Energy without ZPE': int(raw_data["Reaction energy without ZPE (kJ/mol)"][ce39_idx])}

    for molecule_surface_system in molecule_surface_system_list:
        if molecule_surface_system not in calc_molecule_surface_list:
            calc_molecule_surface_list += [molecule_surface_system]
    
    if system_molecule not in calc_molecule_list:
        calc_molecule_list += [system_molecule]

ce39_dataset_df = pd.DataFrame.from_dict(ce39_dataset_dict, orient='index')
display(ce39_dataset_df)

,Reaction,MAD Weight,Reactant Molecule,Reactant Surface,Products,Reaction Enthalpy,Reaction Energy,Reaction Energy without ZPE
01,CO + Ni(111) → CO/Ni(111),1.00,[CO],[Ni_fcc_111_2x2],[CO-Ni_fcc_111_2x2],-122,-119,-124
02,CO + Pt(111) → CO/Pt(111),1.00,[CO],[Pt_fcc_111_2x2],[CO-Pt_fcc_111_2x2],-120,-117,-124
03,CO + Pd(111) → CO/Pd(111),1.00,[CO],[Pd_fcc_111_2x2],[CO-Pd_fcc_111_2x2],-143,-139,-144
04,CO + Pd(100) → CO/Pd(100),1.00,[CO],[Pd_fcc_100_2x2],[CO-Pd_fcc_100_2x2],-155,-151,-157
05,CO + Rh(111) → CO/Rh(111),1.00,[CO],[Rh_fcc_111_2x2],[CO-Rh_fcc_111_2x2],-139,-135,-142
06,CO + Ir(111) → CO/Ir(111),1.00,[CO],[Ir_fcc_111_2x2],[CO-Ir_fcc_111_2x2],-158,-154,-164
07,CO + Cu(111) → CO/Cu(111),1.00,[CO],[Cu_fcc_111_2x2],[CO-Cu_fcc_111_2x2],-53,-52,-57
08,CO + Ru(001) → CO/Ru(001),1.00,[CO],[Ru_hcp_001_2x2],[CO-Ru_hcp_001_2x2],-158,-154,-161
09,CO + Co(001) → CO/Co(001),1.00,[CO],[Co_hcp_001_2x2],[CO-Co_hcp_001_2x2],-115,-112,-119
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),0.50,[NO],[Ni_fcc_100_2x2],"[N-Ni_fcc_100_2x2, O-Ni_fcc_100_2x2]",-290,-288,-299


## DFT estimates

### BEEF-vdW

In [3]:
# Calculate the adsorption energy from BEEF-vdW with a 4L and 6L slab
ce39_eads_dict = {ce39_idx: {'Reaction': '', 'Experiment': 0, 'BEEFvdW (4L)': 0, 'BEEFvdW (6L)': 0} for ce39_idx in ce39_dataset_dict}

for method in ['4L','6L']:
    if method == '4L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_4L'
    elif method == '6L':
        root_folder = f'Data/CE39/Eads_BEEFvdW_6L'

    for ce39_idx in ce39_dataset_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        for product_idx in range(num_products):
            total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/{method}_{products[product_idx]}/OUTCAR.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/{method}_{reactant_surface}/OUTCAR.gz', code='vasp')
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
        else:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR.gz', code='vasp')
            reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/{method}_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/{method}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
            ce39_eads_dict[ce39_idx][f'BEEFvdW ({method})'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

        ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
        ce39_eads_dict[ce39_idx]['Reaction'] = ce39_dataset_dict[ce39_idx]['Reaction']

# Convert to DataFrame
ce39_eads_df = pd.DataFrame.from_dict(ce39_eads_dict, orient='index')
mad_dict = {method: np.mean(abs(ce39_eads_df[method] - ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for method in ['BEEFvdW (4L)', 'BEEFvdW (6L)']}
# Add an additional row at the bottom that is the MAD for each method
ce39_eads_df.loc['MAD'] = ['MAD', 0, mad_dict['BEEFvdW (4L)'], mad_dict['BEEFvdW (6L)']]

# Round to nearest integer and convert to integer
ce39_eads_df = ce39_eads_df.round(0).astype({'Experiment': 'Int64', 'BEEFvdW (4L)': 'Int64', 'BEEFvdW (6L)': 'Int64', 'Reaction': str}) 

display(ce39_eads_df)

,Reaction,Experiment,BEEFvdW (4L),BEEFvdW (6L)
01,CO + Ni(111) → CO/Ni(111),-124,-150,-151
02,CO + Pt(111) → CO/Pt(111),-124,-133,-135
03,CO + Pd(111) → CO/Pd(111),-144,-158,-148
04,CO + Pd(100) → CO/Pd(100),-157,-152,-150
05,CO + Rh(111) → CO/Rh(111),-142,-161,-163
06,CO + Ir(111) → CO/Ir(111),-164,-172,-179
07,CO + Cu(111) → CO/Cu(111),-57,-52,-50
08,CO + Ru(001) → CO/Ru(001),-161,-162,-156
09,CO + Co(001) → CO/Co(001),-119,-135,-137
10,NO + Ni(100) → N/Ni(100) + O/Ni(100),-299,-393,-385


## Comparison between NSC-DFAs, RPA and DFT

In [4]:
# Analyse the interaction energy calculations
ce39_int_dict = {ce39_idx: {'RPA': 0 , 'BEEFvdW': 0, 'hBEEFvdW': 0,'dhBEEFvdW': 0} for ce39_idx in ce39_dataset_dict}
final_ce39_eads_dict = {
    ce39_idx:
        {
            'Reaction': '',
            'Experiment': 0,
            'RPA': 0,
            'BEEFvdW': 0,
            'hBEEFvdW': 0,
            'dhBEEFvdW': 0,
        }
        | {dft_xc: 0 for dft_xc in dft_xc_list}
    for ce39_idx in ce39_dataset_dict
}

# Start by analyzing BEEFvdW
root_folder = "Data/CE39/Eint_BEEFvdW"
for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)

    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR.gz', code='vasp')
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR.gz', code='vasp') 
        ce39_int_dict[ce39_idx]['BEEFvdW'] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


# Analyze hBEEFvdW, dhBEEFvdW and RPA
root_folder = f'Data/CE39/Eint_hBEEFvdW'
rpa_root_folder = f'Data/CE39/Eint_RPA'

for ce39_idx in ce39_int_dict:
    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)
    calc_energy_dict = {
        '00': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '01': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '02': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0},
        '03_3': {'total_product_energy': 0, 'reactant_molecule_energy': 0, 'reactant_surface_energy': 0, 'eint': 0}
    }
    # hBEEF part
    for calc_type in calc_energy_dict:
        for product_idx in range(num_products):
            calc_energy_dict[calc_type]['total_product_energy'] += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{calc_type}.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['eint'] = (calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - num_products * calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV
        else:
            calc_energy_dict[calc_type]['reactant_molecule_energy'] = get_energy(f'{root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp')
            calc_energy_dict[calc_type]['reactant_surface_energy'] = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{calc_type}.gz', code='vasp') 
            calc_energy_dict[calc_type]['eint'] = (2 / 9 * calc_energy_dict[calc_type]['total_product_energy'] - calc_energy_dict[calc_type]['reactant_molecule_energy'] - calc_energy_dict[calc_type]['reactant_surface_energy']) * 1000 / kjmol_to_meV

    # RPA correlation part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz', code='vasp_rpa')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        rpac_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa')- (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz', code='vasp_rpa') 
        rpac_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV

    # RPA EXX part
    total_product_energy = 0
    for product_idx in range(num_products):
        total_product_energy += get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_02_EXX.gz', code='vasp')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = get_energy(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_02_EXX.gz', code='vasp')
        rpax_eint = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV
    else:
        reactant_molecule_energy = get_energy(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp')
        reactant_surface_energy = (1/3)*get_energy(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') - (1/9)*get_energy(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_02_EXX.gz', code='vasp') 
        rpax_eint = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV


    beefx_eint = calc_energy_dict['02']['eint']
    exx_eint = calc_energy_dict['03_3']['eint']
    nlc_eint = calc_energy_dict['00']['eint'] - calc_energy_dict['01']['eint']
    beefc_eint = calc_energy_dict['01']['eint'] - calc_energy_dict['02']['eint']
    dh_x_frac = 0.25
    dh_c_frac = 0.15
    ce39_int_dict[ce39_idx]['dhBEEFvdW'] = dh_x_frac * exx_eint + (1-dh_x_frac)*beefx_eint + dh_c_frac*rpac_eint + (1-dh_c_frac)*beefc_eint + nlc_eint
    h_x_frac = 0.175
    ce39_int_dict[ce39_idx]['hBEEFvdW'] = h_x_frac * exx_eint + (1-h_x_frac)*beefx_eint + beefc_eint + nlc_eint
    ce39_int_dict[ce39_idx]['RPA'] = rpax_eint + rpac_eint

    final_ce39_eads_dict[ce39_idx]['Experiment'] = ce39_dataset_dict[ce39_idx]['Reaction Energy without ZPE']
    final_ce39_eads_dict[ce39_idx]['Reaction'] = r'\ce{' + ce39_dataset_dict[ce39_idx]['Reaction'] + r'}'


# Convert interaction energy to adsorption energy

for method in ['RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW']:
    for ce39_idx in ce39_int_dict:
        final_ce39_eads_dict[ce39_idx][method] = ce39_int_dict[ce39_idx][method] - ce39_int_dict[ce39_idx]['BEEFvdW'] + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)']

# Calculate the adsorption energy for each DFT XC functional and convert to adsorption energy using the interaction energy from BEEF-vdW
for dft_xc in dft_xc_list:
    root_folder = f'Data/CE39/Eads_DFT_4L'
    for ce39_idx in final_ce39_eads_dict:
        reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
        reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
        products = ce39_dataset_dict[ce39_idx]['Products']
        sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
        num_products = len(products)

        total_product_energy = 0
        for product_idx in range(num_products):
            total_product_energy += get_energy(f'{root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_{dft_xc}.gz', code='vasp')

        if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
            reactant_surface_energy = get_energy(f'{root_folder}/Surface/4L_{reactant_surface}/OUTCAR_{dft_xc}.gz', code='vasp')
            final_ce39_eads_dict[ce39_idx][dft_xc] = (total_product_energy - reactant_molecule_energy - num_products * reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']
        else:
            reactant_molecule_energy = get_energy(f'{root_folder}/Molecule/{reactant_molecule}/OUTCAR_{dft_xc}.gz', code='vasp')
            reactant_surface_energy = (1/3)*get_energy(f'{root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp')  - (1/9)*get_energy(f'{root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_{dft_xc}.gz', code='vasp') 
            final_ce39_eads_dict[ce39_idx][dft_xc] = (2 / 9 * total_product_energy - reactant_molecule_energy - reactant_surface_energy) * 1000 / kjmol_to_meV + ce39_eads_dict[ce39_idx]['BEEFvdW (6L)'] - ce39_eads_dict[ce39_idx]['BEEFvdW (4L)']


In [5]:
metrics_methods = ['RPA', 'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW'] + dft_xc_list

chemisorption_list = [f'{ce39_idx:02d}' for ce39_idx in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]]
physisorption_list = [f'{ce39_idx:02d}' for ce39_idx in [26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]]

# Combine final_ce39_eads_dict and ce39_dft_xc_eads_dict into a single DataFrame
final_ce39_eads_df = pd.DataFrame.from_dict(final_ce39_eads_dict, orient='index')

# MAD
total_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df[m] - final_ce39_eads_df['Experiment']) * ce39_dataset_df['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['T'] = ['MAD (Total)', 0] + [total_mad_dict_final[m] for m in metrics_methods]

# MAD Physisorption
physisorption_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df.loc[physisorption_list][m] - final_ce39_eads_df.loc[physisorption_list]['Experiment']) * ce39_dataset_df.loc[physisorption_list]['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['P'] = ['MAD (Physisorption)', 0] + [physisorption_mad_dict_final[m] for m in metrics_methods]

# MAD Chemisorption
chemisorption_mad_dict_final = {m: np.mean(np.abs(final_ce39_eads_df.loc[chemisorption_list][m] - final_ce39_eads_df.loc[chemisorption_list]['Experiment']) * ce39_dataset_df.loc[chemisorption_list]['MAD Weight']) for m in metrics_methods}
final_ce39_eads_df.loc['C'] = ['MAD (Chemisorption)', 0] + [chemisorption_mad_dict_final[m] for m in metrics_methods]

# Round and cast
astype_map = {'Experiment': 'Int64', 'Reaction': str}
astype_map.update({m: 'Int64' for m in metrics_methods})
final_ce39_eads_df = final_ce39_eads_df.round(0).astype(astype_map)

# Update column order to be 'Reaction', 'Experiment', 'dhBEEFvdW', 'hBEEFvdW', 'RPA', 'BEEFvdW' followed by the DFT XC functionals in the order of dft_xc_list
final_ce39_eads_df = final_ce39_eads_df[['Reaction', 'Experiment', 'dhBEEFvdW', 'hBEEFvdW', 'RPA', 'BEEFvdW'] + dft_xc_list]
# Change the column names to be more readable
final_ce39_eads_df.columns = ['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW'] + [method_dict[dft_xc]['label'] for dft_xc in dft_xc_list]

# Make a shortened final_ce39_eads_df with just ['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW']
final_ce39_eads_df_nsc = final_ce39_eads_df[['Reaction', 'Experiment', 'dhBEEF-vdW@BEEF-vdW', 'hBEEF-vdW@BEEF-vdW', 'RPA@PBE', 'BEEF-vdW']]
# Make another shortened final_ce39_eads_df with just ['Reaction', 'Experiment'] + dft_xc_list[:8]
final_ce39_eads_df_dft = final_ce39_eads_df[['Experiment'] + [method_dict[dft_xc]['label'] for dft_xc in dft_xc_list]]

display(final_ce39_eads_df)

convert_df_to_latex_input(
    df = final_ce39_eads_df_nsc,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_eads_1",
    caption = r"Adsorption energies for the CE39 dataset calculated with dhBEEF-vdW@BEEF-vdW, hBEEF-vdW@BEEF-vdW, RPA@PBE, BEEF-vdW compared against experiment. All values are in kJ/mol.",
    replace_input= {
        "[t]": "[c]",
        r"→" : r"$\rightarrow$",
        r" & \rotatebox{90}{Reaction}": r"\rotatebox{90}{Index} & Reaction",
        r" OPt(111)": r" O/Pt(111)"
    },
    index = True,
    rotate_column_header=True,
    longtable=True,
    float_fmt="%.0f",
    filename = "Tables/SI_Table-CE39_Adsorption_Energies_NSC.tex"
)

convert_df_to_latex_input(
    df = final_ce39_eads_df_dft,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_eads_2",
    caption = r"Adsorption energies for the CE39 dataset calculated with several density functional approximation compared against experiment. All values are in kJ/mol.",
    replace_input= {
        "[t]": "[c]",
        r"→" : r"$\rightarrow$",
        r" & \rotatebox{90}{Experiment}": r"\rotatebox{90}{Index}  & \rotatebox{90}{Experiment}"
    },
    index = True,
    rotate_column_header=True,
    longtable=True,
    float_fmt="%.0f",
    filename = "Tables/SI_Table-CE39_Adsorption_Energies_DFA.tex"
)


,Reaction,Experiment,dhBEEF-vdW@BEEF-vdW,hBEEF-vdW@BEEF-vdW,RPA@PBE,BEEF-vdW,PBE,PBEsol,RPBE,revPBE,BLYP,r$^2$SCAN,MS2,revTPSS,PBE-D3,PBE-DdsC,RPBE-D3,optPBE-vdW,revvdW-DF2,r$^2$SCAN+rVV10
01,\ce{CO + Ni(111) → CO/Ni(111)},-124,-140,-139,-100,-151,-220,-229,-143,-187,-126,-186,-148,-174,-200,-200,-206,-179,-200,-202
02,\ce{CO + Pt(111) → CO/Pt(111)},-124,-147,-155,-135,-135,-165,-205,-133,-137,-110,-182,-176,-164,-197,-183,-213,-160,-180,-198
03,\ce{CO + Pd(111) → CO/Pd(111)},-144,-149,-164,-136,-148,-182,-229,-143,-148,-123,-191,-172,-174,-202,-197,-213,-176,-199,-207
04,\ce{CO + Pd(100) → CO/Pd(100)},-157,-153,-162,-144,-150,-180,-220,-146,-150,-132,-192,-179,-170,-202,-194,-208,-176,-193,-206
05,\ce{CO + Rh(111) → CO/Rh(111)},-142,-170,-178,-140,-163,-185,-215,-157,-160,-145,-197,-182,-178,-204,-202,-206,-184,-196,-211
06,\ce{CO + Ir(111) → CO/Ir(111)},-164,-188,-196,-159,-179,-198,-228,-171,-174,-158,-213,-189,-190,-220,-212,-222,-198,-209,-227
07,\ce{CO + Cu(111) → CO/Cu(111)},-57,-52,-44,-44,-50,-70,-99,-44,-47,-33,-82,-69,-70,-92,-87,-109,-72,-82,-96
08,\ce{CO + Ru(001) → CO/Ru(001)},-161,-162,-167,-142,-156,-177,-208,-150,-153,-134,-186,-175,-173,-199,-189,-213,-176,-189,-201
09,\ce{CO + Co(001) → CO/Co(001)},-119,-134,-120,-106,-137,-161,-197,-130,-134,-118,-161,-188,-159,-182,-178,-203,-163,-178,-176
10,\ce{NO + Ni(100) → N/Ni(100) + O/Ni(100)},-299,-393,-405,-407,-385,-505,-508,-369,-451,-382,-491,-535,-450,-453,-445,-442,-449,-481,-511


## Computational Cost

In [6]:
# Get computational cost data
ce39_cost_dict = {ce39_idx: {'BEEF-vdW': 0, 'EXX':0, 'RPA':0} for ce39_idx in ce39_dataset_dict}
ce39_ratio_dict = {ce39_idx: {'BEEF-vdW': 1, 'EXX':0, 'RPA':0} for ce39_idx in ce39_dataset_dict}

for ce39_idx in ce39_cost_dict:
    exx_root_folder = f'Data/CE39/Eint_hBEEFvdW'
    rpa_root_folder = f'Data/CE39/Eint_RPA'
    beef_root_folder = f'Data/CE39/Eint_hBEEFvdW'

    reactant_molecule = ce39_dataset_dict[ce39_idx]['Reactant Molecule'][0]
    reactant_surface = ce39_dataset_dict[ce39_idx]['Reactant Surface'][0]
    products = ce39_dataset_dict[ce39_idx]['Products']
    sc_size = products[0].split('_')[-1]  # Extract the supercell size from the first product
    num_products = len(products)

    # BEEF-vdW cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_00.gz')

    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_00.gz')
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_00.gz')
    else:
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_00.gz')
        total_cost += get_vasp_walltime(f'{beef_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_00.gz') + get_vasp_walltime(f'{beef_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_00.gz')
    ce39_cost_dict[ce39_idx]['BEEF-vdW'] = (total_cost/3600)*192

    # EXX cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_03_3.gz')
    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_03_3.gz')
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_03_3.gz')
    else:
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz')
        total_cost += get_vasp_walltime(f'{exx_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz') + get_vasp_walltime(f'{exx_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_03_3.gz')
    ce39_cost_dict[ce39_idx]['EXX'] = (total_cost/3600)*192

    # RPA cost
    total_cost = 0
    for product_idx in range(num_products):
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule-Surface/4L_{products[product_idx]}/OUTCAR_04_RPA.gz')
    if 'H2OOH' not in ce39_dataset_dict[ce39_idx]['Reaction']:
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_{reactant_surface}/OUTCAR_04_RPA.gz')
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Surface/4L_{reactant_surface}/OUTCAR_04_RPA.gz')
    else:
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule/4L_{reactant_molecule}_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz')
        total_cost += get_vasp_walltime(f'{rpa_root_folder}/Molecule-Surface/4L_O-Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz') + get_vasp_walltime(f'{rpa_root_folder}/Surface/4L_Pt_fcc_111_{sc_size}/OUTCAR_04_RPA.gz')
    ce39_cost_dict[ce39_idx]['RPA'] = (total_cost/3600)*192
    
for ce39_idx in ce39_ratio_dict:
    ce39_ratio_dict[ce39_idx]['EXX'] = ce39_cost_dict[ce39_idx]['EXX'] / ce39_cost_dict[ce39_idx]['BEEF-vdW']
    ce39_ratio_dict[ce39_idx]['RPA'] = ce39_cost_dict[ce39_idx]['RPA'] / ce39_cost_dict[ce39_idx]['BEEF-vdW']

# Convert to DataFrame and display
ce39_ratio_df = pd.DataFrame.from_dict(ce39_ratio_dict, orient='index')
# Round to nearest integer
ce39_ratio_df = ce39_ratio_df.round(0).astype(int)
# Change the index columns to be the reaction string instead of the index
ce39_ratio_df.index = ce39_ratio_df.index.map(lambda x: r'\ce{' + ce39_dataset_dict[x]['Reaction'] + r'}')
# Add name to the index
ce39_ratio_df.index.name = 'Reaction'
display(ce39_ratio_df)

convert_df_to_latex_input(
    df=ce39_ratio_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:ce39_cost_ratio",
    caption = r"Ratio between the computational cost of EXX and RPA calculations compared to BEEF-vdW for the CE39 dataset.",
    longtable=True,
    replace_input= {
        r'→': r'$\rightarrow$',
        r' OPt(111)': r' O/Pt(111)'
    },
    filename = "Tables/SI_Table-CE39_Cost_Ratio.tex")

,BEEF-vdW,EXX,RPA
Reaction,,,
\ce{CO + Ni(111) → CO/Ni(111)},1,18,24
\ce{CO + Pt(111) → CO/Pt(111)},1,23,36
\ce{CO + Pd(111) → CO/Pd(111)},1,25,45
\ce{CO + Pd(100) → CO/Pd(100)},1,27,56
\ce{CO + Rh(111) → CO/Rh(111)},1,18,52
\ce{CO + Ir(111) → CO/Ir(111)},1,15,29
\ce{CO + Cu(111) → CO/Cu(111)},1,22,35
\ce{CO + Ru(001) → CO/Ru(001)},1,14,73
\ce{CO + Co(001) → CO/Co(001)},1,18,24


In [15]:
# Updated 7-method list
methods_to_plot = ['01_PBE','03_RPBE','07_MS2','12_optPBEvdW','14_r2SCANrVV10',
                   'BEEFvdW', 'hBEEFvdW', 'dhBEEFvdW','RPA']

# 7 colors
colors = ['blue', 'orange', 'green', 'red', 'purple', 'cyan', 'magenta','black', 'gray']

fig, axs = plt.subplots(4, 2, figsize=(6.559,5.9), dpi=600, sharex='col', constrained_layout=True, height_ratios=[1,0.5,1,3.5])

bar_width = 0.6
x = np.arange(len(methods_to_plot))

# grab the GridSpec
gs = axs[0,0].get_gridspec()

# remove the two axes in the first column
axs[0,0].remove()
axs[1,0].remove()

# create one big axis spanning rows 0–1, column 0
ax_chem = fig.add_subplot(gs[:2, 0])

# plot wmad_dict_chem
ax_chem.bar(
    x,
    [chemisorption_mad_dict_final[m] for m in methods_to_plot],
    color=[color_dict[method_dict[m]['color']] for m in methods_to_plot],
    width=bar_width
)

ax_chem.set_title("Chemisorption", fontsize=10)


for i, v in enumerate([chemisorption_mad_dict_final[m] for m in methods_to_plot]):
    ax_chem.text(i, v + 2, f'{v:.1f}',
                 ha='center', fontsize=8, fontweight='bold')
    
ax_chem.set_xticklabels([])

########################################
# PANEL 3 — PHYSISORPTION
########################################
axs[2,0].bar(x, [physisorption_mad_dict_final[m] for m in methods_to_plot],
           color=[color_dict[method_dict[m]['color']] for m in methods_to_plot], width=bar_width)
axs[2,0].set_title("Physisorption", fontsize=10)

for i, v in enumerate([physisorption_mad_dict_final[m] for m in methods_to_plot]):
    axs[2,0].text(i, v + 2, f'{v:.1f}', ha='center', fontsize=8, fontweight='bold')

########################################
# PANEL 4 — TOTAL CE39 (now LAST)
########################################
for m_idx, m in enumerate(methods_to_plot):
    axs[3,0].bar(x[m_idx], total_mad_dict_final[m],
               color=color_dict[method_dict[m]['color']], width=bar_width, label = method_dict[m]['label'])


# axs[3,0].bar(x, [wmad_dict_final[m] for m in methods_to_plot],
#            color=colors, width=bar_width)
axs[3,0].set_title("Total", fontsize=10)

for i, v in enumerate([total_mad_dict_final[m] for m in methods_to_plot]):
    axs[3,0].text(i, v + 1, f'{v:.1f}', ha='center', fontsize=8, fontweight='bold')

########################################
# Shared X-axis
########################################
axs[3,0].set_xticks(x)
for i, v in enumerate([m for m in methods_to_plot]):
    axs[3,0].text(i+0.03, 1, method_dict[v]['label'], ha='center', fontsize=8, fontweight='bold',rotation = 90)

# Add shaded square from 0 to 12 on both axes
axs[3,1].fill_between([0, 20], 0, 20, color=color_dict['yellow'], alpha=0.6, edgecolor = 'none', label = '< 20 kJ/mol')
axs[3,1].fill_between([0, 13], 0, 13, color=color_dict['grey'], alpha=0.6, edgecolor = 'none', label = '< 13 kJ/mol')

chemisorption_mad_list = np.array([chemisorption_mad_dict_final[m] for m in method_dict])
physisorption_mad_list = np.array([physisorption_mad_dict_final[m] for m in method_dict])


axs[3,1].scatter(chemisorption_mad_list, physisorption_mad_list, edgecolors='k', facecolors=[color_dict[method_dict[m]['color']] for m in method_dict], s=30)
for method in method_dict:
    if method == 'dhBEEFvdW':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 13, physisorption_mad_dict_final[method] - 4, method_dict[method]['label'], fontsize=7, fontweight='bold', color='black')
    elif method == 'hBEEFvdW':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 16, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7, fontweight='bold', color='black')
    elif method == '03_RPBE':
        axs[3,1].text(chemisorption_mad_dict_final[method] - 8.5, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7)
    else:
        axs[3,1].text(chemisorption_mad_dict_final[method] + 1.5, physisorption_mad_dict_final[method] - 1, method_dict[method]['label'], fontsize=7)

axs[3,1].set_xlim(0, 80)
axs[3,1].set_ylim(0, 80)
axs[3,1].set_xticks(np.arange(0, 81, 20))
axs[3,1].set_yticks(np.arange(0, 81, 20))

axs[3,0].set_xticklabels([])

ax_chem.set_ylim([0,75])
ax_chem.set_yticks([0,25,50,75])
axs[2,0].set_ylim([0,75])
axs[2,0].set_yticks([0,25,50,75])
axs[3,0].set_ylim([0,50])

axs[0,1].axis('off')
axs[1,1].axis('off')
axs[2,1].axis('off')

fig.supylabel('Mean Absolute Deviation (kJ/mol)', fontsize=12)

plt.savefig("Figures/MAIN_Figure-CE39_Comparison.png")

# CO adsorption

In [18]:
# Get layer effect
layer_rel_ene_dict = {x: {method: 0  for method in ['4L','5L','Rel']} for x in ['Cu','Pt','Rh','Pd']}

root_folder = 'Data/CO_Ads/Geom_Effect'

for metal in ['Cu','Pt','Rh','Pd']:
    atop_energy = get_energy(f'{root_folder}/{metal}111_4L/ATOP/OUTCAR_00.gz',code='vasp')
    fcc_energy = get_energy(f'{root_folder}/{metal}111_4L/FCC/OUTCAR_00.gz',code='vasp')
    layer_rel_ene_dict[metal]['4L'] = (atop_energy - fcc_energy) * 1000 / kjmol_to_meV
    atop_energy = get_energy(f'{root_folder}/{metal}111_5L/ATOP/OUTCAR_00.gz',code='vasp')
    fcc_energy = get_energy(f'{root_folder}/{metal}111_5L/FCC/OUTCAR_00.gz',code='vasp')
    layer_rel_ene_dict[metal]['5L'] = (atop_energy - fcc_energy) * 1000 / kjmol_to_meV
    layer_rel_ene_dict[metal]['Rel'] = layer_rel_ene_dict[metal]['5L'] - layer_rel_ene_dict[metal]['4L']


co_ads_dict = {x: {method: 0  for method in ['RPA','BEEFvdW','hBEEFvdW','dhBEEFvdW'] + dft_xc_list} for x in ['Cu','Pt','Rh','Pd']}